<a href="https://colab.research.google.com/github/genaiconference/Agentic_KAG_Workshop_DHS_2026/blob/main/06_Comparison_Matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Goal of the Notebook
This notebook is designed to compare different Retrieval Augmented Generation (RAG) approaches, specifically
- Traditional RAG,
- Agentic RAG,
- GraphRAG, and
- Agentic KAG.

The comparison will be conducted by sending a query to each approach and analyzing the quality and relevance of the generated answers. A question bank may be used to create a comparison matrix to systematically evaluate the performance of each method.

In [1]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

Cloning into 'Agentic_KAG_Workshop_DHS_2026'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 115 (delta 59), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 2.95 MiB | 5.83 MiB/s, done.
Resolving deltas: 100% (59/59), done.


## Install Required Packages

In [1]:
!pip install -r /content/Agentic_KAG_Workshop_DHS_2026/requirements.txt --quiet

## Load credentials from .env

In [2]:
import os

os.chdir("/content/Agentic_KAG_Workshop_DHS_2026")

from dotenv import load_dotenv

load_dotenv()  # This loads .env at project root

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# Set OPENAI_API_KEY as env variable for openai/neo4j-graphrag compatibility
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## Initialize the Neo4j Driver
The Neo4j driver allows you to connect and perform read and write transactions with the database.

In [3]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

# (Optional) Test the connection
driver.verify_connectivity()

## Initialize OpenAI LLM and Embeddings via neo4j-graphrag
We will use OpenAI **GPT-5-mini**. The GraphRAG Python package supports any LLM model, including models from OpenAI, Google VertexAI, Anthropic, Cohere, Azure OpenAI, local Ollama models, and any chat model that works with LangChain. You can also implement a custom interface for any other LLM.

Likewise, we will use OpenAI’s **text-embedding-3-small** for the embedding model, but you can use other embedders from different providers.

In [4]:
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

neo4j_llm = OpenAILLM(
    model_name="gpt-5-mini",
)

embedder = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

## Initialize OpenAI LLM and Embeddings via langchain_openai
We will use OpenAI GPT-5-mini and text-embedding-3-small for the embedding model.

In [5]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Initialize OpenAI LLM using LangChain
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,
                 model_name="gpt-5-mini",
                 temperature=0)

# Initialize OpenAI Embedding model using LangChain
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY,
                              model="text-embedding-3-small")

## Initialize Langfuse Handler for Tracing

In [6]:
from langfuse.langchain import CallbackHandler
from langfuse import get_client

os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

langfuse_handler = CallbackHandler()

Langfuse client is authenticated and ready!


## Load data
Load the data from the movie data chunks from text_chunks.pkl file.

In [7]:
import pickle
from langchain_core.documents import Document

with open("text_chunks.pkl", "rb") as f:
    text_chunks = pickle.load(f)

# Convert to Langchain documents
documents = [
    Document(
        page_content=chunk.text,
        metadata={
            **chunk.metadata,
            "uid": chunk.uid,
            "chunk_index": chunk.index,
        },
    )
    for chunk in text_chunks.chunks
]
print(f"Created {len(documents)} LangChain documents.")

Created 4869 LangChain documents.


In [8]:
print(type(documents[0]))
print(documents[0].page_content[:200])
print(documents[0].metadata)

<class 'langchain_core.documents.base.Document'>
Title: Eternals
Release date: 2021-11-03
Runtime (minutes): 156
Budget (USD): 200000000
Revenue (USD): 402064899
Vote average: 6.927
Genres: Science Fiction, Action, Adventure
Production companies: Ma
{'title': 'Eternals', 'release_date': '2021-11-03', 'dataset': 'TMDB+IMDb Movies', 'uid': '37719c70-7e5e-4a6c-84a1-957cfee08305', 'chunk_index': 0}


## Traditional RAG

Uses both BM25 (sparse) and vector (dense) retrieval.

Chain combines retrieved docs and generates answer via LLM.

In [9]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.documents import Document
from urllib.parse import urlparse
from langchain_community.vectorstores import Chroma
from langchain_classic.prompts import ChatPromptTemplate
from IPython.display import display, Markdown

persist_directory = os.getcwd() +'/vectorstore/chroma/'

# Create the vector store
vectordb = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(vectordb._collection.count())

# Setup
similarity_search_retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k": 5})
bm25_retriever = BM25Retriever.from_documents(documents=documents, k=5)
ensemble_retriever = EnsembleRetriever(retrievers=[similarity_search_retriever, bm25_retriever], weights=[0.5, 0.5])


template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
Avoid using generic phrases like "Provide context" or "as per context.

Question: {input}

Context: {context}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

# Combine docs and chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(ensemble_retriever, combine_docs_chain)

def run_traditional_rag(question):
    response = rag_chain.invoke({"input": question}, config={"callbacks": [langfuse_handler]})
    return response["answer"]

/tmp/ipykernel_43373/4257186350.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


14607


## Agentic RAG

An agent tool uses the ensemble retriever, wrapped via a ReAct agent.

Can plan, retry, and reason through sub-steps.

In [10]:
from langchain_classic.agents import create_react_agent, AgentExecutor
import prompts
from langchain_classic.tools import Tool
from langchain_classic.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    PromptTemplate,
)

# Define the retriever as a tool
retriever_tool = Tool(
    name="AV_Agentic_RAG_tool",
    description="Useful for retrieving relevant documents based on input questions.",
    func=lambda query: ensemble_retriever.invoke(query),
)

tools = [retriever_tool]

# Get the ReAct prompt
prompt = prompts.REACT_PROMPT


# Create the ReAct agent
def get_react_agent(llm, tools, system_prompt, verbose=False):
    """Helper function for creating agent executor"""
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            MessagesPlaceholder(variable_name="conversation_history", optional=True),
            HumanMessagePromptTemplate(
                prompt=PromptTemplate(input_variables=["input"], template="{input}")
            ),
            AIMessagePromptTemplate(
                prompt=PromptTemplate(
                    input_variables=["agent_scratchpad"], template="{agent_scratchpad}"
                )
            ),
        ]
    )
    agent = create_react_agent(llm, tools, prompt)
    return AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=verbose,
        stream_runnable=True,
        handle_parsing_errors=True,
        max_iterations=5,
        return_intermediate_steps=True,
    )

generate_agent = get_react_agent(
        llm,
        tools,
        prompt,
        verbose=False,
    )


def run_agentic_rag(question):
    answer = generate_agent.invoke({"input": question}, config={"callbacks": [langfuse_handler]})
    return answer["output"]


## GraphRAG

In [11]:
from neo4j_graphrag.retrievers import Text2CypherRetriever
from neo4j_graphrag.retrievers import HybridCypherRetriever
from neo4j_graphrag.generation import GraphRAG
import prompts
from examples import examples
from neo4j_graphrag.schema import get_schema

# Initialize the Text2Cypher retriever
t2c_retriever = Text2CypherRetriever(
    llm=neo4j_llm,
    neo4j_schema=get_schema(driver),
    driver=driver,
    custom_prompt=prompts.custom_text2cypher_prompt,
    examples=examples
)


def run_graphrag(question):
  response = t2c_retriever.search(query_text=question)
  cypher_query = response.metadata['cypher']
  print(cypher_query)
  hc_retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name="entity_vector_index",
    fulltext_index_name="entity_fulltext_index",
    retrieval_query=cypher_query,
    embedder=embedder,
)
  rag = GraphRAG(retriever=hc_retriever, llm=neo4j_llm)

  # Query the graph
  response = rag.search(query_text=question,
                        retriever_config={"top_k": 10},
                        return_context=True,
                        response_fallback="I can not answer this question because I have no relevant context.",
                        config={"callbacks": [langfuse_handler]}
                        )
  return response.answer

## Agentic KAG

In [12]:
import tools
from IPython.display import Markdown
import json
import agent_utils

def run_agentic_KAG(query):
    """
    Runs the answer generation flow: Text2Cypher -> Hybrid Retrieval -> Final Answer.
    """
    # Run Hybrid Retrieval using all the available tools
    hybrid_agent = agent_utils.get_react_agent(
        llm,
        [tools.av_hybrid_tool, tools.global_retriever_tool, tools.local_retriever_tool],
        prompts.REACT_PROMPT,
        verbose=False
    )
    hybrid_input = json.dumps({
        "query": query,
    })

    print(f"Running Hybrid Retrieval with input: {hybrid_input}")
    answer = hybrid_agent.invoke({"input": hybrid_input}, config={"callbacks": [langfuse_handler]})
    final_answer = answer['output']
    return final_answer

## Compare the approaches

Create a comparison of the four approaches (Traditional RAG, Agentic RAG, GraphRAG, Agentic KAG)


In [13]:
def compare_rag_answers(question: str) -> dict:
    return {
        "question": question,
        "traditional_rag": run_traditional_rag(question),
        "agentic_rag": run_agentic_rag(question),
        "graphrag": run_graphrag(question),
        "agentic_kag": run_agentic_KAG(question),
    }

def show_rag_answers(result: dict):
    """
    Pretty-prints the results from each RAG pipeline in a structured Markdown format.
    """
    sections = [
        ("❓ Question", result["question"]),
        ("🔵 Traditional RAG", result["traditional_rag"]),
        ("🟢 Agentic RAG", result["agentic_rag"]),
        ("🟣 GraphRAG", result["graphrag"]),
        ("🟠 Agentic KAG", result["agentic_kag"]),
    ]

    for title, content in sections:
        display(Markdown(f"## {title}\n\n{content}"))
        display(Markdown("---"))

In [14]:
question_bank = [#"List down all the sessions or workshops which uses LangGraph?",
                 #"when is Arun delivering a session",
                 #"How many sessions are happening? Give me a breakdown by Session Type"
                 #"give me all the sessions that happen between 12-1 on day 1 in which auditorium should I attend Arun's session?",
                 #"I like only sessions related to knowledge graphs tell me when and where and who is delivering such sessions",
                 #"how many tea breaks do we have and when are they ?",
                 #"I am hungry now when do they serve lunch?"
                 #"I am a big lover of ai agents and I am interested in putting things in production. Suggest me a tailor made agenda, mention the name of the Session or workshop along with Instructor name?"
                 "Compare Oppenheimer and Interstellar across director, cast overlap, genres, production companies, awards, runtime, and audience ratings."
 ]

for q in question_bank:
    result = compare_rag_answers(q)
    show_rag_answers(result)

KeyError: "Input to ChatPromptTemplate is missing variables {'SYSTEM_PROMPT'}.  Expected: ['SYSTEM_PROMPT', 'agent_scratchpad', 'input'] Received: ['input', 'intermediate_steps', 'agent_scratchpad']\nNote: if you intended {SYSTEM_PROMPT} to be part of the string and not a variable, please escape it with double curly braces like: '{{SYSTEM_PROMPT}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "

In [15]:
print(type(prompts.REACT_PROMPT))
print(prompts.REACT_PROMPT)

<class 'str'>


### SYSTEM_PROMPT
{SYSTEM_PROMPT}

### Answering and Formatting Instructions

1. **Markdown Formatting (MANDATORY):**
   - All responses must be formatted in Markdown.
   - Use bold text for all the headers and subheaders.
   - Use bullets, tables wherever applicable.
   - Do not use plain text or paragraphs without Markdown structure.
   - Ensure that you use hyphens (-) for list bullets. For sub-bullets, indent using 2 spaces (not tabs). Ensure proper nesting and consistent formatting.

2. **Citations Must (MANDATORY):**
    - Citations must be immediately placed after the relevant content. Cite relevant URLs as meaningful hyperlinks only if provided to you else ignore.
    - Do not place citations at the end or in a separate references section. They should appear directly after the statement being referenced. **Place inline citations immediately after the relevant content**
    - Do not include tool names or retriever names in citations.

### AGENT'S RESPONSE WORKFLO